### batch data processing

In [2]:
from pathlib import Path
import re
import time
import numpy as np
import pandas as pd
import os
import mrcfile

In [3]:
def convert_one_xds_ascii(infile: Path, out_dir: Path) -> Path:
    s = str(infile)
    m = re.search(r'(?:^|[\\/])experiment[-_](\d+)(?:[\\/])', s, flags=re.IGNORECASE)
    if m:
        base = f"exp{m.group(1)}.hkl"
    else:
        p1 = infile.parent.name.replace(' ', '_')
        p2 = infile.parent.parent.name.replace(' ', '_') if infile.parent.parent else 'root'
        base = f"{p2}_{p1}.hkl"

    out_path = out_dir / base

    written = 0
    with infile.open('r', encoding='utf-8', errors='ignore') as fin, \
         out_path.open('w', encoding='utf-8') as fout:
        for line in fin:
            ls = line.lstrip()
            if not ls or ls.startswith('!'):
                continue  
            parts = ls.split()
            if len(parts) < 5:
                continue
            try:
                h, k, l = int(float(parts[0])), int(float(parts[1])), int(float(parts[2]))
                I, sigI = float(parts[3]), float(parts[4])
            except ValueError:
                continue
            fout.write(f"{h:4d} {k:4d} {l:4d} {I:12.2f} {sigI:12.2f}\n")
            written += 1

    if written == 0:
        try:
            out_path.unlink()
        except FileNotFoundError:
            pass
        raise RuntimeError(f"No valid data rows found in {infile}")
    return out_path

def batch_convert_xds_ascii(root: Path, out_dir: Path = None):
    if out_dir is None:
        out_dir = Path.cwd() / "exphkls"
    out_dir.mkdir(parents=True, exist_ok=True)

    files = list(root.rglob("XDS_ASCII.HKL"))
    if not files:
        print(f"⚠️ no XDS_ASCII.HKL under{root}  ")
        return []
    results = []
    print(f"📂 输入根目录: {root}")
    print(f"📦 输出目录  : {out_dir}\n")
    for f in files:
        try:
            print(f"⏳ 处理: {f}")
            outp = convert_one_xds_ascii(f, out_dir)
            print(f"   → 输出: {outp.name}")
            results.append((f, outp))
        except Exception as e:
            print(f"   ❌ 跳过（原因：{e})")
    print(f"\n🎉 完成！成功转换 {len(results)} 个文件。")
    return results
def parse_exp_num(name):
    """从 expN.hkl 提取 N"""
    m = re.match(r"exp(\d+)\.hkl$", name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None
    
def weighted_mean(values, sigmas):
    """加权平均 I；若 σI 不可用则普通平均"""
    values, sigmas = np.asarray(values, float), np.asarray(sigmas, float)
    mask = np.isfinite(values) & np.isfinite(sigmas) & (sigmas>0)
    if mask.sum() == 0:
        return np.nan
    w = 1.0 / (sigmas[mask]**2)
    return np.sum(w * values[mask]) / np.sum(w)
def extract_int():
    records = []
    for f in sorted(HKL_DIR.glob("exp*.hkl")):
        exp_num = parse_exp_num(f.name)
        if exp_num is None:
            continue
        df = pd.read_csv(f, delim_whitespace=True, header=None, names=["h","k","l","I","SIGI"])
        h,k,l = TARGET_HKL
        if MERGE_FRIEDEL:
            sel = ((df.h==h)&(df.k==k)&(df.l==l)) | ((df.h==-h)&(df.k==-k)&(df.l==-l))
        else:
            sel = (df.h==h)&(df.k==k)&(df.l==l)
        df_sel = df[sel]
        if not df_sel.empty:
            Imean = weighted_mean(df_sel["I"], df_sel["SIGI"])
            records.append((exp_num, Imean))
    
    if records:
        out_df = pd.DataFrame(sorted(records), columns=["expN","Intensity"])
        out_df.to_csv(outfile, sep="\t", index=False, header=False, float_format="%.4f")
        print(f"✅ 已生成文件: {outfile}")
        print(out_df.head())
    else:
        print(f"⚠️ 没有在 {HKL_DIR} 里找到反射 {TARGET_HKL} 的数据。")

In [32]:
def convert_one_hkl1(infile: Path, out_dir: Path) -> Path:
    """
    Convert XDSCONV output 1.HKL to expN.hkl
    """
    s = str(infile)
    m = re.search(r'(?:^|[\\/])experiment[-_](\d+)(?:[\\/])', s, flags=re.IGNORECASE)
    if m:
        base = f"exp{m.group(1)}.hkl"
    else:
        p1 = infile.parent.name.replace(' ', '_')
        p2 = infile.parent.parent.name.replace(' ', '_') if infile.parent.parent else 'root'
        base = f"{p2}_{p1}.hkl"

    out_path = out_dir / base
    written = 0

    with infile.open('r', encoding='utf-8', errors='ignore') as fin, \
         out_path.open('w', encoding='utf-8') as fout:

        fw = fout.write
        for line in fin:
            if not line.strip():
                continue

            parts = line.split()
            # 1.HKL is strictly: h k l I sigI
            if len(parts) < 5:
                continue

            try:
                h = int(parts[0])
                k = int(parts[1])
                l = int(parts[2])
                I = float(parts[3])
                sigI = float(parts[4])
            except Exception:
                continue

            fw(f"{h:4d} {k:4d} {l:4d} {I:12.4f} {sigI:12.4f}\n")
            written += 1

    if written == 0:
        out_path.unlink(missing_ok=True)
        raise RuntimeError(f"No valid reflections in {infile}")

    return out_path


def batch_convert_hkl1(root: Path, out_dir: Path = None):
    """
    Recursively find 1.HKL and convert to expN.hkl
    """
    if out_dir is None:
        out_dir = Path.cwd() / "exphkls"
    out_dir.mkdir(parents=True, exist_ok=True)

    files = list(root.rglob("1.HKL"))
    if not files:
        print(f"⚠️ no 1.HKL under {root}")
        return []

    results = []
    print(f"📂 输入根目录: {root}")
    print(f"📦 输出目录  : {out_dir}\n")

    for f in sorted(files):
        try:
            print(f"⏳ 处理: {f}")
            outp = convert_one_hkl1(f, out_dir)
            print(f"   → 输出: {outp.name}")
            results.append((f, outp))
        except Exception as e:
            print(f"   ❌ 跳过（原因：{e})")

    print(f"\n🎉 完成！成功转换 {len(results)} 个文件。")
    return results


# ============================================================
#  Part 2. 强度提取
# ============================================================

def parse_exp_num(name):
    """从 expN.hkl 提取 experiment 编号"""
    m = re.match(r"exp(\d+)\.hkl$", name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None


def weighted_mean(values, sigmas):
    """加权平均强度 I"""
    values = np.asarray(values, float)
    sigmas = np.asarray(sigmas, float)
    mask = np.isfinite(values) & np.isfinite(sigmas) & (sigmas > 0)

    if mask.sum() == 0:
        return np.nan

    w = 1.0 / (sigmas[mask] ** 2)
    return np.sum(w * values[mask]) / np.sum(w)


def extract_int(
    HKL_DIR: Path,
    TARGET_HKL=(0, 0, 0),
    MERGE_FRIEDEL=True,
    outfile: Path = Path("intensity_vs_exp.txt")
):
    """
    Extract intensity evolution of a given (hkl)
    """
    records = []
    h0, k0, l0 = TARGET_HKL

    for f in sorted(HKL_DIR.glob("exp*.hkl")):
        exp_num = parse_exp_num(f.name)
        if exp_num is None:
            continue

        I_list, sig_list = [], []

        with f.open("r") as fin:
            for line in fin:
                if not line.strip():
                    continue

                h, k, l, I, sigI = map(float, line.split())
                h, k, l = int(h), int(k), int(l)

                if MERGE_FRIEDEL:
                    if (h, k, l) == (h0, k0, l0) or (h, k, l) == (-h0, -k0, -l0):
                        I_list.append(I)
                        sig_list.append(sigI)
                else:
                    if (h, k, l) == (h0, k0, l0):
                        I_list.append(I)
                        sig_list.append(sigI)

        if I_list:
            Imean = weighted_mean(I_list, sig_list)
            records.append((exp_num, Imean))

    if records:
        df = pd.DataFrame(sorted(records), columns=["expN", "Intensity"])
        df.to_csv(outfile, sep="\t", index=False, header=False, float_format="%.4f")
        print(f"✅ 已生成文件: {outfile}")
        print(df.head())
        return df
    else:
        print(f"⚠️ 未找到反射 {TARGET_HKL}")
        return None

In [4]:
def sum_intensity_in_mrc(path):
    """Return the sum of voxel intensities in an MRC file."""
    with mrcfile.open(path, permissive=True) as mrc:
        data = mrc.data.astype(np.float64)
        return float(np.sum(data))


def compute_dataset_sums(base_dir, inner_mrc_dir="RED"):
    """Compute summed intensity for every experiment_x."""
    results = {}

    for exp_name in sorted(os.listdir(base_dir)):
        exp_path = os.path.join(base_dir, exp_name)
        if not (os.path.isdir(exp_path) and exp_name.startswith("experiment_")):
            continue

        mrc_dir = os.path.join(exp_path, inner_mrc_dir)
        if not os.path.isdir(mrc_dir):
            print(f"Skip {exp_name}: no '{inner_mrc_dir}' folder.")
            continue

        mrc_files = [f for f in os.listdir(mrc_dir) if f.lower().endswith(".mrc")]
        if not mrc_files:
            print(f"Skip {exp_name}: no .mrc files.")
            continue

        print(f"Processing {exp_name}...")

        total_sum = 0.0
        for fname in mrc_files:
            total_sum += sum_intensity_in_mrc(os.path.join(mrc_dir, fname))

        results[exp_name] = total_sum
        print(f"  → sum = {total_sum}")
    return results


def save_results_to_txt(results, output_path):
    """Save sums without scientific notation."""
    with open(output_path, "w") as f:
        f.write("Dataset\tTotal_Intensity\n")
        for exp, val in results.items():
            # {:.0f} keeps full number without decimals
            # {:f} keeps decimals (if needed)
            f.write(f"{exp}\t{val:.0f}\n")  # change to {val:f} if you want decimals

    print(f"\nSaved results to: {output_path}")

In [8]:
#normalization number
BASE_DIR = r"C:\Users\luoyi\Desktop\Data\PdSe2\20260128"
INNER_MRC_DIR = "RED"
OUTPUT_TXT = os.path.join(BASE_DIR, "intensity_sums.txt")

if __name__ == "__main__":
    sums = compute_dataset_sums(BASE_DIR, INNER_MRC_DIR)
    save_results_to_txt(sums, OUTPUT_TXT)

    print("\n=== SUMMARY ===")
    for exp, val in sums.items():
        print(f"{exp}: {val:.6e}")


Processing experiment_1...


C:\Users\luoyi\AppData\Roaming\Python\Python312\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x41 0x44
  warnings.warn(str(err), RuntimeWarning)


  → sum = 15362272.0
Processing experiment_10...
  → sum = 14433484.0
Processing experiment_11...
  → sum = 14348494.0
Processing experiment_12...
  → sum = 14185067.0
Processing experiment_13...
  → sum = 14193900.0
Processing experiment_14...
  → sum = 14208607.0
Processing experiment_15...
  → sum = 14157989.0
Processing experiment_16...
  → sum = 14212033.0
Processing experiment_17...
  → sum = 14273357.0
Processing experiment_18...
  → sum = 14271027.0
Processing experiment_19...
  → sum = 13895030.0
Processing experiment_2...
  → sum = 15102345.0
Processing experiment_3...
  → sum = 14959074.0
Processing experiment_4...
  → sum = 15063513.0
Processing experiment_5...
  → sum = 14707137.0
Processing experiment_6...
  → sum = 14180881.0
Processing experiment_7...
  → sum = 14205838.0
Processing experiment_8...
  → sum = 14546331.0
Processing experiment_9...
  → sum = 14446171.0

Saved results to: C:\Users\luoyi\Desktop\Data\PdSe2\20260128\intensity_sums.txt

=== SUMMARY ===
experim

### extract intensities

In [34]:
root = Path(r"C:\Users\luoyi\Desktop\Data\PdSe2\20260128")

HKL_DIR = Path.cwd() / "exphkl"
batch_convert_hkl1(root, HKL_DIR)

📂 输入根目录: C:\Users\luoyi\Desktop\Data\PdSe2\20260128
📦 输出目录  : C:\Users\luoyi\Desktop\Data\PdSe2\20260128\exphkl

⏳ 处理: C:\Users\luoyi\Desktop\Data\PdSe2\20260128\experiment_1\SMV\1.HKL
   → 输出: exp1.hkl
⏳ 处理: C:\Users\luoyi\Desktop\Data\PdSe2\20260128\experiment_10\SMV\1.HKL
   → 输出: exp10.hkl
⏳ 处理: C:\Users\luoyi\Desktop\Data\PdSe2\20260128\experiment_11\SMV\1.HKL
   → 输出: exp11.hkl
⏳ 处理: C:\Users\luoyi\Desktop\Data\PdSe2\20260128\experiment_12\SMV\1.HKL
   → 输出: exp12.hkl
⏳ 处理: C:\Users\luoyi\Desktop\Data\PdSe2\20260128\experiment_13\SMV\1.HKL
   → 输出: exp13.hkl
⏳ 处理: C:\Users\luoyi\Desktop\Data\PdSe2\20260128\experiment_14\SMV\1.HKL
   → 输出: exp14.hkl
⏳ 处理: C:\Users\luoyi\Desktop\Data\PdSe2\20260128\experiment_15\SMV\1.HKL
   → 输出: exp15.hkl
⏳ 处理: C:\Users\luoyi\Desktop\Data\PdSe2\20260128\experiment_16\SMV\1.HKL
   → 输出: exp16.hkl
⏳ 处理: C:\Users\luoyi\Desktop\Data\PdSe2\20260128\experiment_17\SMV\1.HKL
   → 输出: exp17.hkl
⏳ 处理: C:\Users\luoyi\Desktop\Data\PdSe2\20260128\experiment_1

[(WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/experiment_1/SMV/1.HKL'),
  WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/exphkl/exp1.hkl')),
 (WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/experiment_10/SMV/1.HKL'),
  WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/exphkl/exp10.hkl')),
 (WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/experiment_11/SMV/1.HKL'),
  WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/exphkl/exp11.hkl')),
 (WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/experiment_12/SMV/1.HKL'),
  WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/exphkl/exp12.hkl')),
 (WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/experiment_13/SMV/1.HKL'),
  WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/exphkl/exp13.hkl')),
 (WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/experiment_14/SMV/1.HKL'),
  WindowsPath('C:/Users/luoyi/Desktop/Data/PdSe2/20260128/exphkl/exp14.hkl')),
 (WindowsPath('C:/

In [40]:
extract_int(
    HKL_DIR=HKL_DIR,
    TARGET_HKL=(2, 0, 0),
    MERGE_FRIEDEL=False,
    outfile=Path("200.txt")
)


✅ 已生成文件: 200.txt
   expN  Intensity
0     1   281563.0
1     2   233257.0


,expN,Intensity
0,1,281563.0
1,2,233257.0
